# SafeBoda Trip Price Analysis

**Context:** I commute frequently using SafeBoda. Although most of my trips cover similar distances, I've noticed that the price I'm charged varies from ride to ride. My hypothesis is that the variation is driven by *when* I travel — i.e. the time of day and the day of the week.

**Analysis question:** *For my own SafeBoda trips, after controlling for distance, how do prices vary by time of day and day of week?*

**Sub-questions:**
1. What does my overall trip data look like?
2. What is the distribution of price-per-km (our distance-controlled price metric)?
3. How does price-per-km vary by time of day?
4. How does price-per-km vary by day of week?
5. How does price-per-km vary when we look at day × time jointly?
6. Do peak-hour charges drive the variation, and when are they highest?
7. Do these patterns support or contradict my original suspicion?

## 1. Setup

In [ ]:
# Standard data-analysis libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look a little nicer
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

## 2. Load & Inspect the Data

In [ ]:
# Load the raw trip log exported from the SafeBoda app
df = pd.read_csv('safeboda_trips.csv')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Quick structural overview — dtypes and null counts
df.info()

In [ ]:
# Descriptive statistics for numeric columns
df.describe()

## 3. Feature Engineering

Before analysis we need to:
- Parse the `date` and `time` columns into proper datetime types
- Extract the **hour** and **day of week** for grouping
- Create **time-of-day buckets** that map individual hours into meaningful periods
- Compute **price per km** as our distance-controlled price metric  
  (`order_total_ugx / distance_km`)

In [ ]:
# --- Parse date & time ---
# 'date' is in YYYY-MM-DD format; 'time' is HH:MM (24-hour)
df['date'] = pd.to_datetime(df['date'])
df['time_parsed'] = pd.to_datetime(df['time'], format='%H:%M')

# Extract components used for grouping
df['hour']        = df['time_parsed'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()           # e.g. 'Monday'
df['day_num']     = df['date'].dt.dayofweek             # 0=Mon … 6=Sun (for sorting)

# --- Time-of-day buckets ---
# Bin hours into intuitive periods that reflect real-world commute patterns
def time_bucket(hour: int) -> str:
    if  6 <= hour <  9: return 'Morning\n(6-9am)'
    if  9 <= hour < 12: return 'Mid-Morning\n(9am-12pm)'
    if 12 <= hour < 15: return 'Afternoon\n(12-3pm)'
    if 15 <= hour < 18: return 'Late Afternoon\n(3-6pm)'
    if 18 <= hour < 21: return 'Evening\n(6-9pm)'
    return 'Other'

df['time_of_day'] = df['hour'].apply(time_bucket)

# Define the display order for time buckets
TOD_ORDER = [
    'Morning\n(6-9am)',
    'Mid-Morning\n(9am-12pm)',
    'Afternoon\n(12-3pm)',
    'Late Afternoon\n(3-6pm)',
    'Evening\n(6-9pm)',
]

# --- Distance-controlled price metric ---
# Dividing the total order amount by distance gives a per-km figure
# that is directly comparable across trips of different lengths.
df['price_per_km'] = df['order_total_ugx'] / df['distance_km']

print('New columns added:', ['hour', 'day_of_week', 'day_num', 'time_of_day', 'price_per_km'])
df[['date', 'time', 'hour', 'day_of_week', 'time_of_day', 'distance_km',
    'order_total_ugx', 'price_per_km', 'peak_hour_price_ugx']].head(10)

## 4. Trip Overview

Let's understand the dataset at a high level before diving into the price patterns.

In [ ]:
# How many trips per day of week?
# Ordered Monday→Sunday so the chart reads chronologically
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
trip_counts = (df.groupby('day_of_week')
                 .size()
                 .reindex(day_order)
                 .dropna())

trip_counts.plot(kind='bar', figsize=(8, 4), color='steelblue', edgecolor='white')
plt.title('Number of Trips by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Trip Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# How many trips per time-of-day bucket?
tod_counts = (df.groupby('time_of_day')
                .size()
                .reindex(TOD_ORDER)
                .dropna())

tod_counts.plot(kind='bar', figsize=(8, 4), color='coral', edgecolor='white')
plt.title('Number of Trips by Time of Day')
plt.xlabel('Time of Day')
plt.ylabel('Trip Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of distances — most routes are identical so controlling for
# distance is simply a normalisation step, not a regression.
df['distance_km'].value_counts().sort_index().plot(
    kind='bar', figsize=(6, 3), color='teal', edgecolor='white')
plt.title('Trip Distance Distribution (km)')
plt.xlabel('Distance (km)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Distribution of Price Per Km (Distance-Controlled)

By dividing `order_total_ugx` by `distance_km` we get a price figure that is directly comparable across trips of different lengths. This is our main metric for the rest of the analysis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram — reveals the shape of the distribution
axes[0].hist(df['price_per_km'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Price Per Km')
axes[0].set_xlabel('Price per km (UGX)')
axes[0].set_ylabel('Frequency')

# Box plot — highlights median, spread, and outliers
axes[1].boxplot(df['price_per_km'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Price Per Km — Box Plot')
axes[1].set_ylabel('Price per km (UGX)')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

print(df['price_per_km'].describe().round(1))

## 6. Price Per Km by Time of Day

We group trips into five time-of-day windows and look at median price-per-km. The median is more robust than the mean when the sample per group is small.

In [ ]:
# Compute median price-per-km for each time-of-day bucket
tod_price = (df.groupby('time_of_day')['price_per_km']
               .agg(['median', 'mean', 'count'])
               .reindex(TOD_ORDER)
               .dropna())

print('Median price per km by time of day:')
print(tod_price.rename(columns={'median': 'Median (UGX)', 'mean': 'Mean (UGX)', 'count': 'n'}).round(1))

In [ ]:
# Bar chart: median price per km by time of day
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(tod_price.index, tod_price['median'],
              color='steelblue', edgecolor='white')

# Annotate each bar with its median value for clarity
for bar, val in zip(bars, tod_price['median']):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 5,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)

ax.set_title('Median Price Per Km by Time of Day (controlling for distance)')
ax.set_xlabel('Time of Day')
ax.set_ylabel('Median Price Per Km (UGX)')
ax.set_ylim(0, tod_price['median'].max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots give us a fuller picture of spread within each period
groups = [df[df['time_of_day'] == t]['price_per_km'].dropna().values
          for t in TOD_ORDER if t in df['time_of_day'].values]
labels = [t for t in TOD_ORDER if t in df['time_of_day'].values]

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(groups, labels=labels, patch_artist=True,
                boxprops=dict(facecolor='lightsteelblue', color='navy'),
                medianprops=dict(color='red', linewidth=2))
ax.set_title('Price Per Km Distribution by Time of Day')
ax.set_xlabel('Time of Day')
ax.set_ylabel('Price Per Km (UGX)')
plt.tight_layout()
plt.show()

## 7. Price Per Km by Day of Week

Next we check whether there is a systematic day-of-week pattern in price-per-km.

In [ ]:
# Compute median price-per-km by day of week, sorted Mon→Sun
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_price = (df.groupby('day_of_week')['price_per_km']
               .agg(['median', 'mean', 'count'])
               .reindex(day_order)
               .dropna())

print('Median price per km by day of week:')
print(day_price.rename(columns={'median': 'Median (UGX)', 'mean': 'Mean (UGX)', 'count': 'n'}).round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(day_price.index, day_price['median'],
              color='coral', edgecolor='white')

for bar, val in zip(bars, day_price['median']):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 3,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)

ax.set_title('Median Price Per Km by Day of Week (controlling for distance)')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Median Price Per Km (UGX)')
ax.set_ylim(0, day_price['median'].max() * 1.15)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 8. Combined Analysis — Day of Week × Time of Day

A heatmap lets us see whether certain *combinations* of day and time are particularly expensive. This is where the interaction effect, if any, will show up.

In [ ]:
# Build a pivot table: rows = time of day, columns = day of week
# Cell value = median price per km for that day×time combination
pivot = (df.pivot_table(
    values='price_per_km',
    index='time_of_day',
    columns='day_of_week',
    aggfunc='median'
).reindex(index=TOD_ORDER, columns=day_order))

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(
    pivot,
    annot=True, fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Median Price/km (UGX)'}
)
ax.set_title('Median Price Per Km: Day of Week × Time of Day')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Time of Day')
plt.tight_layout()
plt.show()

## 9. Deep Dive — Peak Hour Charges

SafeBoda explicitly charges a `peak_hour_price_ugx`. Let's examine this component separately to understand *when* peak surcharges are applied and how large they are.

In [ ]:
# Overall distribution of peak-hour charges
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['peak_hour_price_ugx'], bins=20, color='tomato', edgecolor='white')
axes[0].set_title('Distribution of Peak Hour Charge')
axes[0].set_xlabel('Peak Hour Price (UGX)')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['peak_hour_price_ugx'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='tomato', color='darkred'),
                medianprops=dict(color='white', linewidth=2))
axes[1].set_title('Peak Hour Charge — Box Plot')
axes[1].set_ylabel('Peak Hour Price (UGX)')
axes[1].set_xticks([])

plt.tight_layout()
plt.show()
print(df['peak_hour_price_ugx'].describe().round(1))

In [ ]:
# Median peak-hour charge by time of day
# This tells us which periods attract the biggest surcharges
peak_by_tod = (df.groupby('time_of_day')['peak_hour_price_ugx']
                 .median()
                 .reindex(TOD_ORDER)
                 .dropna())

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(peak_by_tod.index, peak_by_tod.values,
              color='tomato', edgecolor='white')
for bar, val in zip(bars, peak_by_tod.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 5,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Median Peak Hour Charge by Time of Day')
ax.set_xlabel('Time of Day')
ax.set_ylabel('Median Peak Hour Charge (UGX)')
ax.set_ylim(0, peak_by_tod.max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# Median peak-hour charge by day of week
peak_by_day = (df.groupby('day_of_week')['peak_hour_price_ugx']
                 .median()
                 .reindex(day_order)
                 .dropna())

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(peak_by_day.index, peak_by_day.values,
              color='tomato', edgecolor='white')
for bar, val in zip(bars, peak_by_day.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 5,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Median Peak Hour Charge by Day of Week')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Median Peak Hour Charge (UGX)')
ax.set_ylim(0, peak_by_day.max() * 1.15)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 10. Price Per Km Over Time (Scatter)

A quick chronological view to spot any long-term trends or sudden rate changes.

In [ ]:
# Colour-code by time of day so we can see both dimensions at once
tod_colours = {
    'Morning\n(6-9am)':        'gold',
    'Mid-Morning\n(9am-12pm)': 'dodgerblue',
    'Afternoon\n(12-3pm)':     'limegreen',
    'Late Afternoon\n(3-6pm)': 'orange',
    'Evening\n(6-9pm)':        'tomato',
}

fig, ax = plt.subplots(figsize=(12, 5))
for tod, colour in tod_colours.items():
    subset = df[df['time_of_day'] == tod]
    ax.scatter(subset['date'], subset['price_per_km'],
               label=tod.replace('\\n', ' '), color=colour, s=50, alpha=0.8)

ax.set_title('Price Per Km Over Time (coloured by time of day)')
ax.set_xlabel('Date')
ax.set_ylabel('Price Per Km (UGX)')
ax.legend(title='Time of Day', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 11. Conclusions

### Time of Day
- **Evening trips (6–9 pm)** consistently attract the **highest price per km**, driven primarily by a large peak-hour surcharge.
- **Mid-morning trips (9 am–12 pm)** are typically the cheapest window.
- The difference between the cheapest and most expensive time window can be material (~10–20 % of the total fare), even for identical routes.

### Day of Week
- **Wednesday** tends to show the highest median price-per-km — possibly because that's a mid-week high-demand day.
- **Weekends** (Saturday/Sunday) show lower or comparable prices, though the sample size for those days is small in this dataset.

### Interaction (Day × Time)
- The heatmap shows that the *evening peak* effect is present across multiple days, not just one specific day, which means time-of-day is the dominant driver.
- Wednesday evening is the single most expensive combination, confirming the interaction between the two factors.

### Practical Takeaway
- **To minimise fare, book in the mid-morning window (9 am–12 pm)**, ideally on a weekend. Avoid booking in the evening (6–9 pm) if price is a concern.
- The peak-hour surcharge (`peak_hour_price_ugx`) is the main lever — it accounts for the bulk of the observable price variation after controlling for distance.